In [1]:
import torch
import testdata
# note: had to move this notebook and testdata.py into 
# the multicor_fa directory to run
from _em import _EM_step_no_private_stable, fit_EM_iter

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

Generate fake data

In [4]:
params = {'d': 15, 'k': [0, 0, 0], 'p': [15, 13, 8], 'n': 5000,'sigsq': [0.3, 0.7, 0.5]}

Y, W, L, Phi = testdata.simulate_data(params, private_var=False, verbose=True)
W_init, L_init, Phi_init = testdata.initialize_params(W, L, Phi, private_var=False)

# need Y to be N x p_all
Y = Y.T

No private factors, so Y = WZ + E


In [5]:
Sigma_hat = Y.T @ Y / Y.shape[0]

Ground truth

In [6]:
for W_m in W:
    print((W_m @ W_m.T)[:5, :5])

tensor([[13.0356, -4.3593, -5.6753,  3.1538, -1.3977],
        [-4.3593, 12.6449,  6.5387, -1.5954,  1.4810],
        [-5.6753,  6.5387, 19.5680, -3.4397,  5.3350],
        [ 3.1538, -1.5954, -3.4397,  9.4418,  0.9068],
        [-1.3977,  1.4810,  5.3350,  0.9068, 21.0501]])
tensor([[ 29.4582,  -8.7467, -11.7917,  -8.3773, -13.8419],
        [ -8.7467,  20.5499,  -2.6326,   8.8456,   8.0912],
        [-11.7917,  -2.6326,  17.6258,  -1.7161,   0.7675],
        [ -8.3773,   8.8456,  -1.7161,  20.0000,   7.6453],
        [-13.8419,   8.0912,   0.7675,   7.6453,  15.5123]])
tensor([[30.9129,  5.1401, 16.1450,  5.1870, 12.4005],
        [ 5.1401,  6.7330,  2.4863, -2.5849, -2.2204],
        [16.1450,  2.4863, 17.1539,  5.1462,  4.6694],
        [ 5.1870, -2.5849,  5.1462, 10.9962,  5.6666],
        [12.4005, -2.2204,  4.6694,  5.6666, 21.0577]])


In [7]:
for Phi_m in Phi:
    print((Phi_m)[:5, :5])

tensor([[0.3000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3000]])
tensor([[0.7000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7000]])
tensor([[0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5000]])


Test EM step with complete data (single case)

In [8]:
W_new, _, Phi_new, _, _ = fit_EM_iter(Y, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000)

In [9]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[12.6058, -4.1604, -5.1819,  2.9421, -1.0091],
        [-4.1604, 12.5118,  6.2624, -1.6081,  0.5222],
        [-5.1819,  6.2624, 19.5533, -3.7292,  4.8286],
        [ 2.9421, -1.6081, -3.7292,  9.3402,  0.5201],
        [-1.0091,  0.5222,  4.8286,  0.5201, 20.7047]])
tensor([[ 30.6255,  -8.7609, -12.5047,  -9.1719, -14.0495],
        [ -8.7609,  19.6086,  -2.1974,   9.0294,   7.8434],
        [-12.5047,  -2.1974,  18.2584,  -1.5848,   0.7499],
        [ -9.1719,   9.0294,  -1.5848,  20.1299,   8.1307],
        [-14.0495,   7.8434,   0.7499,   8.1307,  15.5512]])
tensor([[30.5420,  5.3295, 15.4497,  4.9522, 12.6063],
        [ 5.3295,  6.7203,  2.4976, -2.5656, -2.0300],
        [15.4497,  2.4976, 16.4788,  4.8352,  4.9220],
        [ 4.9522, -2.5656,  4.8352, 11.0106,  5.7308],
        [12.6063, -2.0300,  4.9220,  5.7308, 21.3799]])


In [10]:
for Phi_m in Phi_new:
    print((Phi_m)[:5, :5])

tensor([[0.3026, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3174, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2672, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.3291, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3139]])
tensor([[0.6874, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7148, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.7122, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7015, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7098]])
tensor([[0.4761, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4862, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4898, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4939, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.5024]])


Test EM step with missing data, max one mode missing per any sample

In [30]:
Y_miss = Y.clone().detach()

In [31]:
Y_miss[1000:1025, :params['p'][0]] = float('nan')
Y_miss[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
Y_miss[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')

In [32]:
Y_miss[1020:1030, params['p'][0]-3:params['p'][0]+3]

tensor([[    nan,     nan,     nan, -1.2335, -2.5996,  0.3399],
        [    nan,     nan,     nan,  3.1362,  6.7268, -1.7106],
        [    nan,     nan,     nan,  3.1012, -4.8104, -0.7755],
        [    nan,     nan,     nan, -0.5550, -3.5540, -2.0354],
        [    nan,     nan,     nan, -7.7895, -1.4882,  2.3596],
        [ 3.5423, -1.9825,  5.8958,     nan,     nan,     nan],
        [ 3.2832,  4.4876,  0.5892,     nan,     nan,     nan],
        [-1.2861,  3.2682,  0.7082,     nan,     nan,     nan],
        [-2.9444, -7.9284, -0.4728,     nan,     nan,     nan],
        [-5.1902, -1.5854, -2.5577,     nan,     nan,     nan]])

In [35]:
Y_miss[1070:1080, params['p'][0]+params['p'][1]-3:params['p'][0]+params['p'][1]+3]

tensor([[    nan,     nan,     nan, -7.7370, -1.2699, -3.4708],
        [    nan,     nan,     nan, -0.9160,  0.1929, -1.2371],
        [    nan,     nan,     nan,  0.0897, -0.5430, -5.0213],
        [    nan,     nan,     nan,  0.1278, -5.1178, -2.5324],
        [    nan,     nan,     nan,  6.4881, -1.1571,  1.9055],
        [ 5.4762,  1.4559,  5.1785,     nan,     nan,     nan],
        [-2.6329,  0.9941, -2.4130,     nan,     nan,     nan],
        [ 3.9825,  5.9273,  0.6064,     nan,     nan,     nan],
        [-7.3806,  0.1721, -0.3838,     nan,     nan,     nan],
        [-0.9212,  0.8540,  1.2325,     nan,     nan,     nan]])

In [36]:
Sigma_hat = Y_miss.nan_to_num().T @ Y_miss.nan_to_num() / Y_miss.shape[0]

In [37]:
W_new, _, Phi_new, _, _ = fit_EM_iter(Y_miss, Sigma_hat, W_init, L_init, Phi_init, maxit = 1000, impute_modes = True)

In [38]:
for W_m in W_new:
    print((W_m @ W_m.T)[:5, :5])

tensor([[12.4827, -4.1125, -5.1017,  2.9244, -0.9733],
        [-4.1125, 12.4276,  6.2134, -1.5912,  0.4667],
        [-5.1017,  6.2134, 19.3451, -3.6284,  4.6727],
        [ 2.9244, -1.5912, -3.6284,  9.1433,  0.5903],
        [-0.9733,  0.4667,  4.6727,  0.5903, 20.4352]])
tensor([[ 29.9624,  -8.6021, -12.3086,  -8.9739, -13.7361],
        [ -8.6021,  19.2781,  -2.1656,   8.9379,   7.6790],
        [-12.3086,  -2.1656,  17.9289,  -1.5693,   0.7956],
        [ -8.9739,   8.9379,  -1.5693,  19.8301,   7.9765],
        [-13.7361,   7.6790,   0.7956,   7.9765,  15.2252]])
tensor([[30.3060,  5.3732, 15.3313,  4.8577, 12.4606],
        [ 5.3732,  6.6357,  2.5111, -2.5104, -1.9632],
        [15.3313,  2.5111, 16.2753,  4.7576,  4.8744],
        [ 4.8577, -2.5104,  4.7576, 10.8343,  5.6687],
        [12.4606, -1.9632,  4.8744,  5.6687, 21.0236]])


In [39]:
for Phi_m in Phi_new:
    print(Phi_m[:5, :5])

tensor([[0.3365, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.3081, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4032, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.3558]])
tensor([[0.8490, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.8394, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.8175, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.7433, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.7784]])
tensor([[0.5433, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5430, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5637, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.5767, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.6450]])


Attempt at more systematic testing for complete data case

In [40]:
metrics = {
    # 'WWt_rmse': [],
    'WWt_corr': [],
    # 'Phi_rmse': [],
    'Phi_corr': [],
}
# simulate 1000 runs 
for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]
    
    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    # metrics['WWt_rmse'].append(torch.sqrt(torch.mean((WW_test - WW)**2)).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['Phi_rmse'].append(torch.sqrt(torch.mean((P_test - P)**2)).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


Manual note: 1000 simulations with complete data took ~ 1m 8.6s to run

In [41]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9979
	Min: 0.9949
	Max: 0.999
Phi_corr
	Mean: 0.9949
	Min: 0.9868
	Max: 0.9982


Attempt at more systematic testing for missing data case

In [42]:
metrics = {
    # 'WWt_rmse': [],
    'WWt_corr': [],
    # 'Phi_rmse': [],
    'Phi_corr': [],
}
# simulate 1000 runs 
for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    # insert missing data
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')
    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    # need to fill in NA Sigma values here
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute_modes=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    # metrics['WWt_rmse'].append(torch.sqrt(torch.mean((WW_test - WW)**2)).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['Phi_rmse'].append(torch.sqrt(torch.mean((P_test - P)**2)).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


Manual note: 1000 simulations with missing data took ~ 1m 55.9s

In [43]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9979
	Min: 0.9955
	Max: 0.999
Phi_corr
	Mean: 0.9723
	Min: 0.9349
	Max: 0.9899


Missing data with more than one mode missing in some samples

In [44]:
metrics = {
    # 'WWt_rmse': [],
    'WWt_corr': [],
    # 'Phi_rmse': [],
    'Phi_corr': [],
}
# simulate 1000 runs 
for i in range(1000):
    if (i+1) % 200 == 0:
        print(f"{i+1} simulations completed")

    # generate data
    Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=False)
    Y = Y.T
    # insert missing data
    Y[1000:1025, :params['p'][0]] = float('nan')
    Y[1020:1070, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
    Y[1065:1100, params['p'][0]+params['p'][1]:] = float('nan')
    # initialize params
    W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=False)
    # need to fill in NA Sigma values here
    Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

    # ground truths
    WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
    P = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_true])
    
    # run EM
    W_new, _, Phi_new, _, _ = fit_EM_iter(
        Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute_modes=True
    )
    WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
    P_test = torch.cat([torch.diag(Phi_m) for Phi_m in Phi_new])
    
    # get metrics
    # metrics['WWt_rmse'].append(torch.sqrt(torch.mean((WW_test - WW)**2)).item())
    metrics['WWt_corr'].append(torch.corrcoef(
        torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
    )[0,1].item())
    # metrics['Phi_rmse'].append(torch.sqrt(torch.mean((P_test - P)**2)).item())
    metrics['Phi_corr'].append(torch.corrcoef(
        torch.stack([P, P_test], dim=0)
    )[0,1].item())

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed


1000 simulations ran in 1m 55.3s

In [45]:
for k, v in metrics.items():
    print(k)
    v = torch.tensor(v)
    print(f"\tMean: {round(torch.mean(v).item(), 4)}")
    print(f"\tMin: {round(torch.min(v).item(), 4)}")
    print(f"\tMax: {round(torch.max(v).item(), 4)}")

WWt_corr
	Mean: 0.9979
	Min: 0.9941
	Max: 0.9989
Phi_corr
	Mean: 0.9703
	Min: 0.9173
	Max: 0.9939
